# Sentinel — prompt search on the GPU host

Runs the **Ask the estate** service here rather than on a laptop, so the web app
has something to talk to whether or not anyone's machine is awake.

Run the cells in order. Each one is safe to re-run.

**The service must outlive this notebook.** Everything below starts it with
`setsid`, in its own session, so restarting the kernel or letting the notebook
idle-cull does not take the service with it. Do not "run" the server in a cell
and leave the cell running — that is exactly the arrangement that dies quietly.

## 1. What this host has

The card is shared. This reports what is actually free rather than what is
installed, because the number that matters is the one nobody else is using.

In [ ]:
import os, shutil, subprocess, sys, textwrap

# Where this repository was cloned on the server.
REPO = os.environ.get('SENTINEL_REPO', os.path.expanduser('~/sentinel-command-center'))
SERVER = os.path.join(REPO, 'pipeline', 'ask', 'server')
HOME = os.environ.get('SENTINEL_HOME', os.path.expanduser('~/sentinel'))
os.environ['SENTINEL_HOME'] = HOME

def sh(cmd, **kw):
    """Run a command and stream its output into the notebook."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True,
                         executable='/bin/bash', **kw)
    for line in p.stdout:
        print(line, end='')
    return p.wait()

print('repo    :', REPO, '' if os.path.isdir(REPO) else '  <-- NOT FOUND')
print('install :', HOME)
print()
sh('nvidia-smi --query-gpu=index,name,memory.total,memory.free,utilization.gpu '
   '--format=csv')
print()
print('cores:', os.cpu_count())
sh("free -g | awk 'NR<=2'")

If the repo path is wrong, clone it and set `SENTINEL_REPO`:

```bash
git clone https://github.com/ashitrai04/Gujarat-Police-Hackathon.git ~/sentinel-command-center
```

## 2. Set it up (once)

Creates a virtualenv, installs torch and the rest, fetches Ollama into your home
directory — no root — and pulls the models. Ten to twenty minutes the first time,
almost all of it download.

In [ ]:
# A 40 GB card is worth a better vision-language model than a laptop can hold.
# 7B parses and verifies noticeably better than 3B and is still small here.
os.environ['ASK_VLM_MODEL'] = 'qwen2.5vl:7b'

sh(f'bash {SERVER}/setup.sh')

## 3. Footage

Put the recordings on this host, then index them. Upload with Jupyter's file
browser, or `scp` / `rsync` them in:

```bash
rsync -av --progress "D:/React folder/feed_recordings/" user@tbiaiserver:~/sentinel/footage/
```

In [ ]:
os.makedirs(f'{HOME}/footage', exist_ok=True)
sh(f'ls -lh {HOME}/footage | head -30; echo; '
   f'echo "videos: $(find {HOME}/footage -name \'*.mp4\' | wc -l)"')

## 4. Build the index

Indexing is **decode-bound, not GPU-bound** — every frame is decoded and one per
second kept — so on 256 cores this is far quicker than it was on a laptop. About
13 minutes for an hour of 1080p there; expect a small fraction of that here.

`--every 0.5` doubles the keyframes and costs almost nothing, because the decode
happens either way.

In [ ]:
sh(f'bash {SERVER}/build_index.sh {HOME}/footage --every 1.0')

## 5. Start it

Detached, with a watchdog that restarts the service if it dies and restarts
Ollama if *it* dies. On a shared host another job can take the memory this one
wanted and kill it mid-afternoon with nobody watching.

In [ ]:
sh(f'bash {SERVER}/run.sh')

## 6. Check it answers

Straight against the service, before involving the browser — if this works and
the web app does not, the problem is reachability, not the service.

In [ ]:
import json, urllib.request

TOKEN = open(f'{HOME}/token').read().strip()
BASE = 'http://127.0.0.1:8077'

def ask(prompt, k=4, verify=False):
    body = json.dumps({'prompt': prompt, 'k': k, 'verify': verify}).encode()
    req = urllib.request.Request(f'{BASE}/ask', data=body, headers={
        'content-type': 'application/json', 'x-ask-token': TOKEN})
    return json.loads(urllib.request.urlopen(req, timeout=300).read())

h = urllib.request.Request(f'{BASE}/health', headers={'x-ask-token': TOKEN})
print(json.loads(urllib.request.urlopen(h, timeout=30).read()))

r = ask('a busy road junction full of traffic')
print('\nparser      :', r['plan'].get('parser'))
print('candidates  :', r['n_candidates'], 'of', r['n_indexed'])
print('timing      :', r['timing'])
for x in r['results']:
    print(f"  {x['score']:+.4f}  {x['camera_id']:>10s}  t+{x['t_s']:.0f}s  {x['counts']}")

## 7. Let the web app reach it

Three ways, best first.

**a. A hostname the browser can resolve.** If this host is reachable from where
you browse — the same network, a VPN, a campus DNS name — point the app straight
at it. No tunnel, no third party, nothing to expire.

**b. Jupyter's own proxy.** If the only way in is the Jupyter URL, `jupyter-server-proxy`
exposes the port under that same origin, so it inherits Jupyter's authentication:

```bash
pip install jupyter-server-proxy   # then restart the Jupyter server
# the service then answers at  <jupyter-url>/proxy/8077/
```

**c. ngrok**, if neither of those is possible. Reserve a free domain so the
address survives restarts, then `bash run.sh --tunnel`.

Whichever you choose, put the address and token in Vercel as `VITE_ASK_API_URL`
and `VITE_ASK_TOKEN` and redeploy once.

In [ ]:
sh(f'bash {SERVER}/run.sh --status')
print()
sh("hostname -I; hostname -f")
print('\ntoken:', TOKEN)

## 8. Day to day

| | |
|---|---|
| `bash run.sh --status` | is it up |
| `bash run.sh --logs` | follow the log |
| `bash run.sh --stop` | stop the service, watchdog and tunnel |
| `bash run.sh` | start again |
| `bash build_index.sh <dir>` | re-index after adding footage |

The watchdog keeps it alive across crashes and across Ollama dying. It does
**not** survive a reboot of the host — if this box restarts, run `run.sh` again
(or ask whoever administers it for a systemd unit, which is the proper fix and
needs root).

In [ ]:
sh(f'tail -40 {HOME}/ask.log')